# ARISTA lineage and generated/observed snapshots — CytoBridge API

**Objective.** Reproduce the ARISTA spatial snapshots and lineage Sankey using only the installed `CytoBridge` public API. Success means the run creates observed/generated panels, a reusable classifier cache, Sankey HTML/SVG/PDF/PNG, and a manifest while importing neither `DeepRUOT` nor `vendor/legacy_arista_stack`.


## Plan

1. Resolve either the portable published checkpoint or a newly trained current checkpoint.
2. Run package interpolation with piecewise spatial warp and classifier KNN=1.
3. Export observed/generated snapshots and lineage Sankey.
4. Verify the manifest and the package-only runtime contract.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path
import sys
import torch

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'downstream_helpers').exists():
    REPO_ROOT = REPO_ROOT.parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from downstream_helpers.arista_api import (
    AristaSpatiotemporalConfig,
    assert_package_only_runtime,
    run_arista_spatiotemporal_api,
)
from downstream_helpers.runner import display_html_outputs, display_svg_outputs

SEED = 42
DEVICE = os.environ.get('CYTOBRIDGE_DEVICE', 'cuda' if torch.cuda.is_available() else 'cpu')
SMOKE = os.environ.get('CYTOBRIDGE_SMOKE', '0') == '1'
MODEL_FORMAT = os.environ.get('ARISTA_MODEL_FORMAT', 'legacy')
ALIGNED_H5AD = os.environ.get('ARISTA_ALIGNED_H5AD') or None
MODEL_DIR = os.environ.get('ARISTA_MODEL_DIR') or None
DEVICE, SMOKE, MODEL_FORMAT


## Configuration

The default portable mode uses the published legacy checkpoint through `CytoBridge.tl.load_legacy_dynamical_model_from_dir`. To evaluate a newly trained current model, set `ARISTA_MODEL_FORMAT=current`, `ARISTA_ALIGNED_H5AD`, and `ARISTA_MODEL_DIR` before starting Jupyter. These are input definitions only; both modes use the same current package simulation and plotting APIs.


In [ ]:
if MODEL_FORMAT == 'current' and (ALIGNED_H5AD is None or MODEL_DIR is None):
    raise ValueError('Current mode requires ARISTA_ALIGNED_H5AD and ARISTA_MODEL_DIR.')

config = AristaSpatiotemporalConfig(
    output_name='arista_lineage_snapshot_api' + ('_smoke' if SMOKE else ''),
    aligned_h5ad=ALIGNED_H5AD,
    model_dir=MODEL_DIR,
    model_format=MODEL_FORMAT,
    time_points=(0.0, 1.0) if SMOKE else (0.0, 1.0, 2.0, 3.0, 4.0),
    interp_time_points=(0.5,) if SMOKE else (0.5, 1.5, 2.5, 3.5),
    plot_3d_time_points=(0.0, 0.5, 1.0),
    n_samples=16 if SMOKE else 7668,
    classifier_epochs=2 if SMOKE else 1000,
    classifier_knn_neighbors=1,
    spatial_warp_to_observed_piecewise=True,
    split_sde_dt=0.1 if SMOKE else 0.01,
    random_seed=SEED,
    device=DEVICE,
    run_communication=False,
    run_3d=False,
)
config


In [ ]:
result = run_arista_spatiotemporal_api(config)
assert_package_only_runtime()
result


## Results

The snapshot mosaic alternates observed and generated slices at observed timepoints and shows generated intermediate slices separately. Smoke mode validates wiring only; its 16 particles and 2 classifier epochs are not quantitative results.


In [ ]:
display_svg_outputs([result.snapshots_dir / 'timepoint_mosaic.svg'])
display_html_outputs([result.lineage_html], height=850)
print(result.manifest_path.read_text(encoding='utf-8'))


## Next checks

- Compare the current-model mosaic with the portable published-checkpoint mosaic using the same seed and 7,668 particles.
- Record classifier balanced accuracy and inspect rare lineage labels before interpreting Sankey widths.
- Use `arista_spatiotemporal_3d_api.ipynb` for attention-derived interactions and the focus-anchor 3D panel.
